# Debugging Earth Engine Layer Visibility

This notebook troubleshoots why the AI layer (Earth DNA / Embedding) is not visible in the Streamlit app.
We will isolate the Earth Engine logic from Streamlit to verify data availability, band names, and visualization parameters.

In [1]:
import ee
import geemap

# Initialize Earth Engine
# This will use the credentials from `earthengine authenticate`
try:
    ee.Initialize()
    print("Earth Engine initialized successfully.")
except Exception as e:
    print(f"Initialization failed: {e}")
    # If using a service account, you might need to authenticate differently here.

Initialization failed: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.


In [ ]:
# Define Region of Interest (ROI)
# Coordinates for Shanghai (Lat: 30.90, Lon: 121.93) used in the app
lat = 30.90
lon = 121.93
point = ee.Geometry.Point([lon, lat])
roi = point.buffer(5000) # Buffer to see more context
print(f"ROI defined at {lat}, {lon}")

In [ ]:
# Define the dataset
dataset_id = 'GOOGLE/RESEARCH/SATELLITE_EMBEDDING/V1/ANNUAL' # app.py uses GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL but let's check both or start with what app.py has
# Actually, let's use what app.py has
dataset_id = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"

collection = ee.ImageCollection(dataset_id).filterBounds(roi)

print(f"Collection size (unfiltered by date): {collection.size().getInfo()}")

# Print available Time ranges
dates = collection.aggregate_array('system:time_start')
readable_dates = [
    ee.Date(d).format('YYYY-MM-dd').getInfo() 
    for d in dates.getInfo()
]
print(f"Available dates: {readable_dates}")

# Check for 2024 specifically as per app.py
filtered_2024 = collection.filterDate('2024-01-01', '2024-12-31')
print(f"Collection size (2024): {filtered_2024.size().getInfo()}")


In [ ]:
# Inspect Metadata
if collection.size().getInfo() > 0:
    first_image = collection.first()
    band_names = first_image.bandNames().getInfo()
    print(f"Band names: {band_names}")
    
    # Try creating a map
    Map = geemap.Map(center=[30.90, 121.93], zoom=10)
    
    # Use the visualization parameters from app.py
    vis_params = {"min": -0.1, "max": 0.1, "gamma": 1.6}
    
    # If 2024 is available, use it. Otherwise use the latest available.
    start_date = '2024-01-01'
    end_date = '2024-12-31'
    
    target_image = collection.filterDate(start_date, end_date).first()
    if target_image.getInfo() is None:
        print(f"No image found for {start_date} to {end_date}. Using the latest available.")
        target_image = collection.sort('system:time_start', False).first()
    
    Map.addLayer(target_image, vis_params, "AI Layer (Debug)")
    Map
else:
    print("Collection is empty.")
